# 02 · Buildings, 3D massing and population

**Question:** How do building assumptions change a synthetic population allocation?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Have Astra critique floor-area proxies and design a conservation test; compare the suggested allocation to the exact baseline.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Allocate people using residential floor area
Change `population_total` or `floor_height_m` and rerun. Integer largest-remainder allocation conserves population.
Floor counts and occupancy are invented; height is a visualization assumption.


In [ ]:
c=city(); population_total=1200; floor_height_m=3.
pop=allocate(c['capacity'],population_total)
flat=allocate(c['width']*c['depth']*c['residential'],population_total)
assert pop.sum()==population_total and (pop[~c['residential']]==0).all()
assert (pop>=0).all()
print('Population conserved:',pop.sum(),'| people reassigned when ignoring floors:',int(abs(pop-flat).sum()/2))


In [ ]:
fig=plt.figure(figsize=(12,5)); ax=fig.add_subplot(121,projection='3d')
ax.bar3d(c['xy'][:,0],c['xy'][:,1],np.zeros(len(pop)),c['width'],c['depth'],c['floors']*floor_height_m,
         color=np.where(c['residential'],'#087f8c','#f2a541'),shade=True)
ax.set(xlabel='East (m)',ylabel='North (m)',zlabel='Height (m)',title='Synthetic 3D massing')
ax=fig.add_subplot(122); dots=ax.scatter(*c['xy'].T,c=pop,s=35,cmap='viridis')
map_axes(ax,'Allocated residents'); fig.colorbar(dots,ax=ax,label='People'); plt.tight_layout(); plt.show()


In [ ]:
rows=[dict(building=f'b{i:03d}',population=int(v),floor_area_m2=float(c['capacity'][i])) for i,v in enumerate(pop)]
export_json('02_population.json',dict(synthetic=True,total=population_total,buildings=rows))
print('Astra prompt: Explain why a floor-area-weighted population surface does not establish household size, poverty, or disease risk. Propose three external validation datasets.')


## Extend with Codex or Astra
Add neighborhood totals and allocate within each neighborhood. Test conservation at both levels. Replace guessed floors with documented height evidence and uncertainty ranges.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
